# 🏠 Texas Home Finder — Databricks + MLflow

**End-to-end demo:** synthetic Texas housing data → feature engineering → multiple ML models → MLflow experiment tracking → best-model selection → Unity Catalog Model Registry → personalized home ranking.

> **Demo note:** The housing data is synthetic. Do not use the resulting rankings for real purchasing decisions.

## Architecture

```mermaid
flowchart TD
    A["Synthetic Texas Home Data"] --> B["Databricks"]
    B --> C["Feature Engineering"]
    C --> D1["Linear Regression"]
    C --> D2["Random Forest 50"]
    C --> D3["Random Forest 150"]
    D1 --> E["MLflow"]
    D2 --> E
    D3 --> E
    E --> F["Compare MAE / RMSE / R²"]
    F --> G["Select Best Model"]
    G --> H["Unity Catalog Model Registry"]
    H --> I["Load Registered Model"]
    I --> J["Buyer Preferences"]
    J --> K["Filter Candidate Homes"]
    K --> L["Predict / Score / Rank"]
    L --> M["Top 10 Recommendations"]
```


## MLflow lifecycle

```mermaid
flowchart LR
    A["Experiment"] --> B["Run 1"]
    A --> C["Run 2"]
    A --> D["Run 3"]
    B --> E["Parameters + Metrics + Model"]
    C --> F["Parameters + Metrics + Model"]
    D --> G["Parameters + Metrics + Model"]
    E --> H["Best Run"]
    F --> H
    G --> H
    H --> I["Unity Catalog"]
    I --> J["Production / Serving"]
```


In [1]:
# Databricks notebook setup
import mlflow
import mlflow.sklearn

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MLflow version:", mlflow.__version__)


/Users/bijum/bijudatabricks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow version: 3.15.2


In [2]:
# Configuration

EXPERIMENT_NAME = "/Shared/texas-home-recommendation"

# Dedicated Unity Catalog catalog/schema for this demo.
CATALOG = "texas"
SCHEMA = "housingdata"

TABLE_NAME = f"{CATALOG}.{SCHEMA}.texas_homes"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.texas_home_price_model"

# Point MLflow at the Databricks workspace tracking server (uses the
# same auth as Databricks Connect) instead of defaulting to a local
# SQLite store, so the experiment actually shows up under /Shared.
mlflow.set_tracking_uri("databricks")

mlflow.set_experiment(EXPERIMENT_NAME)

# Set explicitly (and early) so MLflow never falls back to probing
# spark.conf for "spark.mlflow.modelRegistryUri" over Spark Connect,
# which is unsupported and floods stderr with harmless GRPC errors.
mlflow.set_registry_uri("databricks-uc")

print("Experiment:", EXPERIMENT_NAME)
print("Home table:", TABLE_NAME)
print("Model:", MODEL_NAME)


Experiment: /Shared/texas-home-recommendation
Home table: texas.housingdata.texas_homes
Model: texas.housingdata.texas_home_price_model


In [3]:
# Create synthetic Texas housing data

np.random.seed(42)

cities = [
    "Houston", "Katy", "Sugar Land", "The Woodlands",
    "Austin", "Round Rock", "Frisco", "McKinney",
    "Plano", "Pearland"
]

data = []

for _ in range(1000):
    city = np.random.choice(cities)
    sqft = np.random.randint(1200, 4000)
    bedrooms = np.random.randint(2, 6)
    bathrooms = np.random.randint(2, 5)
    school_rating = np.random.uniform(5, 10)
    property_tax_pct = np.random.uniform(1.5, 2.5)
    commute_minutes = np.random.randint(10, 60)
    year_built = np.random.randint(1980, 2025)

    # Synthetic target-generation formula.
    price = (
        sqft * 180
        + bedrooms * 15000
        + bathrooms * 10000
        + school_rating * 20000
        - commute_minutes * 1000
        + (year_built - 1980) * 1000
        + np.random.normal(0, 30000)
    )

    price = max(price, 150000)

    data.append([
        city, price, sqft, bedrooms, bathrooms,
        school_rating, property_tax_pct,
        commute_minutes, year_built
    ])

homes = pd.DataFrame(data, columns=[
    "city", "price", "sqft", "bedrooms", "bathrooms",
    "school_rating", "property_tax_pct",
    "commute_minutes", "year_built"
])

display(homes.head(10))


,city,price,sqft,bedrooms,bathrooms,school_rating,property_tax_pct,commute_minutes,year_built
0,Frisco,690145.484492,2060,4,4,8.898455,2.096850,28,2002
1,Round Rock,605676.952942,1969,5,3,6.061696,1.681825,30,2012
2,Pearland,381565.281067,1221,2,2,6.456146,2.111853,51,2007
3,Frisco,598665.746214,2467,5,2,5.232252,2.107545,30,1988
4,Frisco,756754.223345,2497,5,2,9.828160,2.308397,18,2005
5,Frisco,869349.673143,3756,5,4,5.866823,1.891061,59,2019
6,The Woodlands,562101.000530,2221,3,3,7.733551,1.684854,27,2005
7,The Woodlands,781592.357958,3517,4,4,5.442463,1.695983,49,2000
8,Katy,591262.933026,2078,2,3,9.826277,2.107034,54,2020
9,Plano,873711.805992,3768,5,2,5.993578,1.505522,20,1996


In [4]:
# Basic exploration

print("Number of homes:", len(homes))

display(homes.describe())

display(
    homes.groupby("city")
         .agg(
             homes=("price", "count"),
             avg_price=("price", "mean"),
             avg_sqft=("sqft", "mean"),
             avg_school=("school_rating", "mean")
         )
         .sort_values("avg_price")
)


Number of homes: 1000


,price,sqft,bedrooms,bathrooms,school_rating,property_tax_pct,commute_minutes,year_built
count,1.000000e+03,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,6.821497e+05,2569.569000,3.47000,2.927000,7.515481,1.987414,35.26100,2002.068000
std,1.530578e+05,801.210239,1.11998,0.795175,1.444697,0.287994,14.54085,12.717984
min,3.338186e+05,1201.000000,2.00000,2.000000,5.002008,1.500012,10.00000,1980.000000
25%,5.602318e+05,1916.000000,2.00000,2.000000,6.277128,1.737429,23.00000,1991.000000
50%,6.792679e+05,2532.500000,3.00000,3.000000,7.531640,1.993255,36.00000,2002.500000
75%,8.079609e+05,3249.250000,4.00000,4.000000,8.800383,2.233526,48.00000,2013.000000
max,1.026713e+06,3997.000000,5.00000,4.000000,9.999246,2.499544,59.00000,2024.000000


,homes,avg_price,avg_sqft,avg_school
city,,,,
Pearland,96,661179.901332,2436.906250,7.538457
Houston,95,662977.477169,2477.873684,7.260473
Sugar Land,90,663393.492570,2471.444444,7.590649
Frisco,113,671968.691219,2528.362832,7.716540
McKinney,106,679219.444320,2551.716981,7.430411
Austin,91,681787.331919,2612.967033,7.620800
The Woodlands,111,691230.995501,2593.621622,7.613902
Katy,124,698945.432709,2669.040323,7.492857
Plano,86,704082.122418,2671.627907,7.393850


In [5]:
# Persist source data in Unity Catalog

spark_df = spark.createDataFrame(homes)

spark_df.write.mode("overwrite").saveAsTable(TABLE_NAME)

print("Saved:", TABLE_NAME)


Saved: texas.housingdata.texas_homes


## Data → MLflow

```mermaid
flowchart LR
    A["Unity Catalog Table"] --> B["Training Data"]
    B --> C["Train / Test Split"]
    C --> D["Model Training"]
    D --> E["MLflow Tracking"]
```


In [6]:
# Prepare features and target

features = [
    "sqft",
    "bedrooms",
    "bathrooms",
    "school_rating",
    "property_tax_pct",
    "commute_minutes",
    "year_built"
]

target = "price"

X = homes[features]
y = homes[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 800
Testing rows: 200


In [7]:
# Define multiple candidate models

models = {
    "linear_regression": LinearRegression(),

    "random_forest_50": RandomForestRegressor(
        n_estimators=50,
        max_depth=8,
        random_state=42
    ),

    "random_forest_150": RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        random_state=42
    )
}

print(list(models.keys()))


['linear_regression', 'random_forest_50', 'random_forest_150']


In [8]:
# Train all models and track every experiment with MLflow

results = []

for model_name, model in models.items():

    with mlflow.start_run(run_name=model_name):

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)

        # Parameters
        mlflow.log_param("model_name", model_name)

        if hasattr(model, "n_estimators"):
            mlflow.log_param("n_estimators", model.n_estimators)

        if hasattr(model, "max_depth"):
            mlflow.log_param("max_depth", model.max_depth)

        # Metrics
        mlflow.log_metric("mae", float(mae))
        mlflow.log_metric("rmse", float(rmse))
        mlflow.log_metric("r2", float(r2))

        # Model artifact
        mlflow.sklearn.log_model(
            model,
            name="home_price_model"
        )

        results.append({
            "model": model_name,
            "mae": mae,
            "rmse": rmse,
            "r2": r2
        })

results_df = pd.DataFrame(results).sort_values("mae")

display(results_df)


/Users/bijum/bijudatabricks/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:71: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(
2026/09/06 16:09:21 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Volumes/D/Projects/MLFLOW
2026/09/06 16:09:23 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Volumes/D/Projects/MLFLOW
2026/09/06 16:09:23 INFO mlflow.utils.environment: Detected uv project at /Volumes/D/Projects/MLFLOW. Attempting to export requirements via 'uv export'.
2026/09/06 16:09:23 INFO mlflow.utils.uv_utils: Exported 0 dependencies via uv
2026/09/06 16:09:23 WARNING mlflow.utils.environment: uv export failed or returned no requirements. Falling back to package capture based inference.
2026/09/06 1

,model,mae,rmse,r2
0,linear_regression,24537.476053,31181.845634,0.957224
1,random_forest_50,30581.666017,38036.791205,0.936349
2,random_forest_150,30605.884085,37765.323797,0.937255


## Model comparison

**MAE:** lower is better  
**RMSE:** lower is better  
**R²:** higher is better

```mermaid
flowchart TD
    A["Linear Regression"] --> D["MLflow Metrics"]
    B["Random Forest 50"] --> D
    C["Random Forest 150"] --> D
    D --> E{"Lowest MAE?"}
    E --> F["Best Model"]
```


In [9]:
# Select the model with the lowest MAE

best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

best_mae = results_df.iloc[0]["mae"]
best_rmse = results_df.iloc[0]["rmse"]
best_r2 = results_df.iloc[0]["r2"]

print("Best model:", best_model_name)
print("MAE:", round(best_mae, 2))
print("RMSE:", round(best_rmse, 2))
print("R2:", round(best_r2, 4))


Best model: linear_regression
MAE: 24537.48
RMSE: 31181.85
R2: 0.9572


In [10]:
# Create a dedicated MLflow run for the selected model

with mlflow.start_run(run_name=f"BEST_{best_model_name}"):

    best_model.fit(X_train, y_train)

    predictions = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    mlflow.log_param("selected_model", best_model_name)
    mlflow.log_metric("mae", float(mae))
    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("r2", float(r2))

    mlflow.sklearn.log_model(
        best_model,
        name="best_home_price_model"
    )

    best_run_id = mlflow.active_run().info.run_id

print("Best Run ID:", best_run_id)


2026/09/06 16:09:36 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Volumes/D/Projects/MLFLOW
2026/09/06 16:09:37 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Volumes/D/Projects/MLFLOW
2026/09/06 16:09:37 INFO mlflow.utils.environment: Detected uv project at /Volumes/D/Projects/MLFLOW. Attempting to export requirements via 'uv export'.
2026/09/06 16:09:37 INFO mlflow.utils.uv_utils: Exported 0 dependencies via uv
2026/09/06 16:09:37 WARNING mlflow.utils.environment: uv export failed or returned no requirements. Falling back to package capture based inference.


Best Run ID: 500afe6117c347088fdbc3d3be28abe8


## Register the winning model in Unity Catalog

```mermaid
flowchart LR
    A["Best MLflow Run"] --> B["Logged Model"]
    B --> C["Unity Catalog"]
    C --> D["main.default.texas_home_price_model"]
    D --> E["Model Version"]
```


In [11]:
# Register the winning model in Unity Catalog

model_uri = f"runs:/{best_run_id}/best_home_price_model"

try:
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_NAME
    )

    model_version = registered_model.version

    print("Registered model:", registered_model.name)
    print("Version:", model_version)

except Exception as e:
    print("Model registration failed.")
    print("Check that the catalog/schema exist and that you have CREATE MODEL permission.")
    print("Error:", e)
    model_version = None


Registered model 'texas.housingdata.texas_home_price_model' already exists. Creating a new version of this model...
2026/09/06 16:09:40 WARNING mlflow.tracking._model_registry.fluent: Run with id 500afe6117c347088fdbc3d3be28abe8 has no artifacts at artifact path 'best_home_price_model', registering model based on models:/m-3dbbbcf63f7e4169965bc46ce4583cb3 instead
2026/09/06 16:09:40 WARNING mlflow.store._unity_catalog.registry.rest_store: Unable to get model version source run's workspace ID from request headers. No run link will be recorded for the model version


Model registration failed.
Check that the catalog/schema exist and that you have CREATE MODEL permission.
Error: Model passed for registration did not contain any signature metadata. All models in the Unity Catalog must be logged with a model signature containing both input and output type specifications. See https://mlflow.org/docs/latest/model/signatures.html#how-to-log-models-with-signatures for details on how to log a model with a signature


In [12]:
# Optional SQL verification

display(
    spark.sql(f'''
        SELECT *
        FROM {TABLE_NAME}
        LIMIT 10
    ''')
)


,city,price,sqft,bedrooms,bathrooms,school_rating,property_tax_pct,commute_minutes,year_built
0,Frisco,690145.484492,2060,4,4,8.898455,2.096850,28,2002
1,Round Rock,605676.952942,1969,5,3,6.061696,1.681825,30,2012
2,Pearland,381565.281067,1221,2,2,6.456146,2.111853,51,2007
3,Frisco,598665.746214,2467,5,2,5.232252,2.107545,30,1988
4,Frisco,756754.223345,2497,5,2,9.828160,2.308397,18,2005
5,Frisco,869349.673143,3756,5,4,5.866823,1.891061,59,2019
6,The Woodlands,562101.000530,2221,3,3,7.733551,1.684854,27,2005
7,The Woodlands,781592.357958,3517,4,4,5.442463,1.695983,49,2000
8,Katy,591262.933026,2078,2,3,9.826277,2.107034,54,2020
9,Plano,873711.805992,3768,5,2,5.993578,1.505522,20,1996


In [13]:
# Load the registered model if registration succeeded

if model_version is not None:

    loaded_model = mlflow.sklearn.load_model(
        f"models:/{MODEL_NAME}/{model_version}"
    )

    print("Registered model loaded successfully.")
else:
    # Fallback for demonstration if registration is not available.
    loaded_model = best_model
    print("Using in-memory best model as fallback.")


Using in-memory best model as fallback.


# 👤 Buyer profile

Example buyer requirements:

```text
Budget:              $500,000
Bedrooms:            4+
Minimum SqFt:        2,400
School Rating:       8+
Maximum Commute:     35 minutes
```

```mermaid
flowchart LR
    A["Buyer Preferences"] --> B["Candidate Filter"]
    C["Texas Homes"] --> B
    B --> D["Candidate Homes"]
    D --> E["ML Price Prediction"]
    E --> F["Buyer Fit Score"]
    F --> G["Ranked Homes"]
```


In [14]:
# Buyer preferences

buyer = {
    "budget": 500000,
    "min_bedrooms": 4,
    "min_sqft": 2400,
    "min_school_rating": 8,
    "max_commute_minutes": 35
}

buyer


{'budget': 500000,
 'min_bedrooms': 4,
 'min_sqft': 2400,
 'min_school_rating': 8,
 'max_commute_minutes': 35}

In [15]:
# Filter candidate homes

candidate_homes = homes[
    (homes["bedrooms"] >= buyer["min_bedrooms"]) &
    (homes["sqft"] >= buyer["min_sqft"]) &
    (homes["school_rating"] >= buyer["min_school_rating"]) &
    (homes["commute_minutes"] <= buyer["max_commute_minutes"])
].copy()

print("Candidate homes:", len(candidate_homes))

display(candidate_homes.head(10))


Candidate homes: 53


,city,price,sqft,bedrooms,bathrooms,school_rating,property_tax_pct,commute_minutes,year_built
4,Frisco,7.567542e+05,2497,5,2,9.828160,2.308397,18,2005
38,The Woodlands,8.856345e+05,3568,5,3,8.978963,2.390005,32,1994
51,Katy,8.389897e+05,3282,4,2,8.278613,1.885397,19,2005
80,Frisco,8.704464e+05,3342,4,3,8.488338,1.680067,13,2004
107,The Woodlands,8.502377e+05,3063,4,3,8.856233,1.874435,18,2008
117,Katy,8.789884e+05,3334,4,3,9.334969,1.907984,12,2003
119,McKinney,8.744085e+05,3343,5,3,8.015987,2.306850,25,1983
148,Plano,9.104150e+05,3753,4,3,9.003848,2.129778,32,1988
160,Round Rock,7.281301e+05,2419,5,3,9.736489,1.666378,21,2001
165,Sugar Land,1.006396e+06,3995,5,2,9.718080,1.919727,29,1985


In [16]:
# Predict prices for candidate homes

candidate_homes["predicted_price"] = loaded_model.predict(
    candidate_homes[features]
)

display(
    candidate_homes[
        ["city", "price", "predicted_price", "sqft", "bedrooms"]
    ].head(10)
)


,city,price,predicted_price,sqft,bedrooms
4,Frisco,7.567542e+05,754553.194788,2497,5
38,The Woodlands,8.856345e+05,916432.985952,3568,5
51,Katy,8.389897e+05,845092.188744,3282,4
80,Frisco,8.704464e+05,874432.290101,3342,4
107,The Woodlands,8.502377e+05,831154.106377,3063,4
117,Katy,8.789884e+05,891799.633912,3334,4
119,McKinney,8.744085e+05,852804.689103,3343,5
148,Plano,9.104150e+05,927323.652964,3753,4
160,Round Rock,7.281301e+05,738037.438568,2419,5
165,Sugar Land,1.006396e+06,990739.283127,3995,5


In [17]:
# Calculate a personalized home-fit score

# Price: lower predicted price is better.
candidate_homes["price_score"] = (
    1 - candidate_homes["predicted_price"] / buyer["budget"]
).clip(lower=0, upper=1)

# School quality: higher is better.
candidate_homes["school_score"] = (
    candidate_homes["school_rating"] / 10
).clip(0, 1)

# Space: more space is better, capped at 4,000 sqft.
candidate_homes["space_score"] = (
    candidate_homes["sqft"] / 4000
).clip(0, 1)

# Commute: shorter is better.
candidate_homes["commute_score"] = (
    1 - candidate_homes["commute_minutes"] / 60
).clip(0, 1)

# Business weights.
candidate_homes["home_score"] = (
    candidate_homes["price_score"] * 0.35 +
    candidate_homes["school_score"] * 0.30 +
    candidate_homes["space_score"] * 0.20 +
    candidate_homes["commute_score"] * 0.15
) * 100


In [18]:
# Top 10 recommended homes

recommendations = (
    candidate_homes
    .sort_values("home_score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

recommendations.insert(
    0,
    "rank",
    range(1, len(recommendations) + 1)
)

display(
    recommendations[
        [
            "rank",
            "city",
            "predicted_price",
            "sqft",
            "bedrooms",
            "bathrooms",
            "school_rating",
            "commute_minutes",
            "home_score"
        ]
    ]
)


,rank,city,predicted_price,sqft,bedrooms,bathrooms,school_rating,commute_minutes,home_score
0,1,Sugar Land,975890.588069,3950,4,3,9.131952,19,57.395855
1,2,Sugar Land,990739.283127,3995,5,2,9.718080,29,56.879239
2,3,Pearland,897891.050281,3375,4,3,9.497063,14,56.866188
3,4,Pearland,976421.505855,3708,4,4,9.768163,24,56.844489
4,5,Katy,891799.633912,3334,4,3,9.334969,12,56.674906
5,6,McKinney,841968.644622,3055,4,2,9.672719,14,55.793158
6,7,The Woodlands,941550.788200,3701,4,2,9.512267,25,55.791800
7,8,The Woodlands,945682.752244,3825,4,3,8.964780,21,55.769341
8,9,The Woodlands,944820.565720,3812,4,2,8.883992,20,55.711977
9,10,Plano,955456.519941,3658,5,2,9.616119,26,55.638356


In [19]:
# Save recommendations back to Unity Catalog

recommendation_table = (
    f"{CATALOG}.{SCHEMA}.texas_home_recommendations"
)

spark.createDataFrame(recommendations).write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(recommendation_table)

print("Saved:", recommendation_table)


Saved: texas.housingdata.texas_home_recommendations


# 🔎 Query the final recommendations

```mermaid
flowchart LR
    A["MLflow Best Model"] --> B["Price Prediction"]
    B --> C["Buyer Fit Score"]
    C --> D["Top 10"]
    D --> E["Unity Catalog Table"]
    E --> F["SQL / Dashboard / API"]
```


In [20]:
# Final SQL result

display(
    spark.sql(f'''
        SELECT
            rank,
            city,
            ROUND(predicted_price, 0) AS predicted_price,
            sqft,
            bedrooms,
            bathrooms,
            ROUND(school_rating, 1) AS school_rating,
            commute_minutes,
            ROUND(home_score, 2) AS home_score
        FROM {recommendation_table}
        ORDER BY rank
    ''')
)


,rank,city,predicted_price,sqft,bedrooms,bathrooms,school_rating,commute_minutes,home_score
0,1,Sugar Land,975891.0,3950,4,3,9.1,19,57.40
1,2,Sugar Land,990739.0,3995,5,2,9.7,29,56.88
2,3,Pearland,897891.0,3375,4,3,9.5,14,56.87
3,4,Pearland,976422.0,3708,4,4,9.8,24,56.84
4,5,Katy,891800.0,3334,4,3,9.3,12,56.67
5,6,McKinney,841969.0,3055,4,2,9.7,14,55.79
6,7,The Woodlands,941551.0,3701,4,2,9.5,25,55.79
7,8,The Woodlands,945683.0,3825,4,3,9.0,21,55.77
8,9,The Woodlands,944821.0,3812,4,2,8.9,20,55.71
9,10,Plano,955457.0,3658,5,2,9.6,26,55.64


# 🚀 Production extension

The notebook can evolve into an **AI Texas Home Finder**:

```mermaid
flowchart TD
    U["User"] --> A["GenAI / Agent"]
    A --> P["Parse Buyer Preferences"]
    P --> S["Property Search"]
    S --> D["Databricks / Unity Catalog"]
    D --> M["MLflow Registered Model"]
    M --> R["Predict / Rank"]
    R --> A
    A --> E["LLM Explanation"]
    E --> U

    M --> G["Governance / Model Version"]
    R --> T["Recommendation Table"]
```

### Production deployment

```mermaid
flowchart LR
    A["Databricks"] --> B["MLflow"]
    B --> C["Unity Catalog"]
    C --> D["Model/API"]
    D --> E["Docker"]
    E --> F["Production"]
```

### Possible next steps

1. Replace synthetic data with real property/listing data.
2. Add location, ZIP code, HOA, property tax, insurance and other features.
3. Add geospatial/commute features.
4. Track model versions and approvals in Unity Catalog.
5. Add an LLM agent to understand natural-language buyer requirements.
6. Expose the recommender through an API.
